## Quality of Care Indicators

Compute district-year quality-of-care indicators from DHIS2 routine data produced by outliers pipelines (`imputed` or `removed`).

Indicators:
- testing_rate = TEST / SUSP
- treatment_rate = MALTREAT / CONF
- case_fatality_rate = MALDTH / MALADM
- prop_adm_malaria = MALADM / ALLADM
- prop_malaria_deaths = MALDTH / ALLDTH
- non_malaria_all_cause_outpatients = ALLOUT (absolute)
- presumed_cases = PRES (absolute)

Stock-out indicators are not implemented yet (on hold, NMDR data pending).

In [ ]:
# Preliminaries
options(scipen = 999)

ROOT_PATH <- "~/workspace"
source(file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "utils", "snt_dhis2_quality_of_care.r"))

snt_environment <- get_setup_variables(SNT_ROOT_PATH = ROOT_PATH, packages = c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "dplyr", "knitr", "scales", "gridExtra"))
config_json <- load_snt_config(snt_environment$CONFIG_PATH, "SNT_config.json")
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
OUTLIERS_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_OUTLIERS_IMPUTATION

PIPELINE_PATH <- file.path(snt_environment$PIPELINES_PATH, "snt_dhis2_quality_of_care")
OUTPUT_DATA_PATH <- file.path(snt_environment$DATA_PATH, "dhis2", "quality_of_care")
REPORT_OUTPUTS_PATH <- file.path(PIPELINE_PATH, "reporting", "outputs")
FIGURES_PATH <- file.path(REPORT_OUTPUTS_PATH, "figures")
dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(REPORT_OUTPUTS_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

In [ ]:
if (!exists("data_action")) data_action <- "imputed"
data_action <- validate_quality_of_care_action(data_action)

indicator_cols <- c("TEST", "SUSP", "MALTREAT", "CONF", "MALDTH", "MALADM", "ALLADM", "ALLDTH", "ALLOUT", "PRES")

routine <- load_dataset_file(
  dataset_id = OUTLIERS_DATASET,
  filename = glue::glue("{COUNTRY_CODE}_routine_outliers_{data_action}.parquet")
)
shapes <- load_dataset_file(
  dataset_id = DHIS2_FORMATTED_DATASET,
  filename = glue::glue("{COUNTRY_CODE}_shapes.geojson")
)

routine <- normalize_qoc_routine_types(routine, indicator_cols = indicator_cols)
qoc     <- aggregate_qoc_district_year(routine, indicator_cols = indicator_cols)

# Derived indicators — edit here to add / remove / modify
if ("TEST"     %in% names(qoc) && "SUSP"   %in% names(qoc)) qoc[, testing_rate     := fifelse(SUSP   > 0, TEST     / SUSP,   NA_real_)]
if ("MALTREAT" %in% names(qoc) && "CONF"   %in% names(qoc)) qoc[, treatment_rate   := fifelse(CONF   > 0, MALTREAT / CONF,   NA_real_)]
if ("MALDTH"   %in% names(qoc) && "MALADM" %in% names(qoc)) qoc[, case_fatality_rate := fifelse(MALADM > 0, MALDTH / MALADM, NA_real_)]
if ("MALADM"   %in% names(qoc) && "ALLADM" %in% names(qoc)) qoc[, prop_adm_malaria := fifelse(ALLADM > 0, MALADM   / ALLADM, NA_real_)]
if ("MALDTH"   %in% names(qoc) && "ALLDTH" %in% names(qoc)) qoc[, prop_malaria_deaths := fifelse(ALLDTH > 0, MALDTH / ALLDTH, NA_real_)]
if ("ALLOUT"   %in% names(qoc)) qoc[, non_malaria_all_cause_outpatients := ALLOUT]
if ("PRES"     %in% names(qoc)) qoc[, presumed_cases := PRES]

qoc <- attach_quality_of_care_shapes(qoc, shapes)

save_quality_of_care_outputs(
  qoc_dt = qoc,
  output_data_path = OUTPUT_DATA_PATH,
  country_code = COUNTRY_CODE,
  data_action = data_action
)

In [ ]:
# Build yearly maps (saved as PNG)
save_quality_of_care_maps(
  qoc_dt = qoc,
  shapes_sf = shapes,
  figures_path = FIGURES_PATH
)